# 22. Robotics action policies — ACT, Diffusion Policy, π0-style flow, FAST

Only width, horizon, and token counts are reduced. ACT, Diffusion Policy, π0-style flow,
and FAST each keep their complete small-scale action-generation path.


In [ ]:
import math
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. ACT — CVAE posterior → action-query decoder → chunk loss


In [ ]:
class TinyACT(nn.Module):
    def __init__(self, obs_dim=8, action_dim=3, chunk=4, d=24, latent=8):
        super().__init__()
        self.latent = latent
        self.obs = nn.Linear(obs_dim, d)
        self.action = nn.Linear(action_dim, d)
        enc = nn.TransformerEncoderLayer(d, 3, 4 * d, batch_first=True)
        dec = nn.TransformerDecoderLayer(d, 3, 4 * d, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, 1)
        self.decoder = nn.TransformerDecoder(dec, 1)
        self.mu = nn.Linear(d, latent)
        self.logvar = nn.Linear(d, latent)
        self.z_proj = nn.Linear(latent, d)
        self.queries = nn.Parameter(torch.randn(1, chunk, d) * 0.02)
        self.head = nn.Linear(d, action_dim)

    def forward(self, obs, target=None):
        b = obs.size(0)
        obs_token = self.obs(obs).unsqueeze(1)
        if target is None:
            mu = torch.zeros(b, self.latent, device=obs.device)
            logvar = torch.zeros_like(mu)
            z = torch.zeros_like(mu)
        else:
            encoded = self.encoder(
                torch.cat([obs_token, self.action(target)], dim=1)
            )[:, 0]
            mu, logvar = self.mu(encoded), self.logvar(encoded)
            z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        memory = torch.cat([obs_token, self.z_proj(z).unsqueeze(1)], dim=1)
        query = self.queries.expand(b, -1, -1)
        return self.head(self.decoder(query, memory)), mu, logvar

obs = torch.randn(4, 8, device=device)
target = torch.randn(4, 4, 3, device=device)
act = TinyACT().to(device)
pred, mu, logvar = act(obs, target)
kl = -0.5 * (1 + logvar - mu.square() - logvar.exp()).mean()
loss = F.l1_loss(pred, target) + 0.005 * kl
loss.backward()
print("ACT:", pred.shape, "inference:", act(obs[:1])[0].shape)
print("ACT grad:", act.head.weight.grad.norm().item())


## 2. Diffusion Policy — conditional temporal U-Net → iterative denoising


In [ ]:
def time_embed(t, d):
    half = d // 2
    freq = torch.exp(
        -math.log(10000.0) * torch.arange(half, device=t.device) / max(half - 1, 1)
    )
    angle = t[:, None] * freq[None]
    return torch.cat([angle.sin(), angle.cos()], dim=-1)

class CondBlock(nn.Module):
    def __init__(self, cin, cout, cdim):
        super().__init__()
        self.c1 = nn.Conv1d(cin, cout, 3, padding=1)
        self.c2 = nn.Conv1d(cout, cout, 3, padding=1)
        self.n1, self.n2 = nn.GroupNorm(4, cout), nn.GroupNorm(4, cout)
        self.cond = nn.Linear(cdim, 2 * cout)
        self.skip = nn.Identity() if cin == cout else nn.Conv1d(cin, cout, 1)

    def forward(self, x, c):
        h = self.n1(self.c1(x))
        scale, shift = self.cond(F.silu(c)).chunk(2, dim=-1)
        h = F.silu(h * (1 + scale[:, :, None]) + shift[:, :, None])
        return F.silu(self.n2(self.c2(h)) + self.skip(x))

class TinyDiffusionPolicy(nn.Module):
    def __init__(self, obs_dim=8, action_dim=3, base=16, cdim=32):
        super().__init__()
        self.cdim = cdim
        self.obs = nn.Linear(obs_dim, cdim)
        self.time = nn.Sequential(nn.Linear(cdim, cdim), nn.SiLU(), nn.Linear(cdim, cdim))
        self.input = nn.Conv1d(action_dim, base, 1)
        self.down = CondBlock(base, base, cdim)
        self.downsample = nn.Conv1d(base, 2 * base, 4, 2, 1)
        self.middle = CondBlock(2 * base, 2 * base, cdim)
        self.upsample = nn.ConvTranspose1d(2 * base, base, 4, 2, 1)
        self.up = CondBlock(2 * base, base, cdim)
        self.output = nn.Conv1d(base, action_dim, 1)

    def forward(self, noisy, obs, t):
        c = self.obs(obs) + self.time(time_embed(t, self.cdim))
        x = self.input(noisy.transpose(1, 2))
        skip = self.down(x, c)
        x = self.middle(self.downsample(skip), c)
        x = self.upsample(x)
        if x.size(-1) != skip.size(-1):
            x = F.interpolate(x, size=skip.size(-1), mode="linear", align_corners=False)
        return self.output(self.up(torch.cat([x, skip], dim=1), c)).transpose(1, 2)

@torch.no_grad()
def sample_diffusion(model, obs, horizon=8, action_dim=3, steps=6):
    x = torch.randn(obs.size(0), horizon, action_dim, device=obs.device)
    for step in range(steps, 0, -1):
        now = torch.full((obs.size(0),), step / steps, device=obs.device)
        nxt = torch.full((obs.size(0),), (step - 1) / steps, device=obs.device)
        eps = model(x, obs, now)
        a = torch.cos(0.5 * math.pi * now)[:, None, None].clamp_min(1e-3)
        s = torch.sin(0.5 * math.pi * now)[:, None, None]
        x0 = (x - s * eps) / a
        an = torch.cos(0.5 * math.pi * nxt)[:, None, None]
        sn = torch.sin(0.5 * math.pi * nxt)[:, None, None]
        x = an * x0 + sn * eps
    return x

dp = TinyDiffusionPolicy().to(device)
clean = torch.randn(4, 8, 3, device=device)
noise = torch.randn_like(clean)
t = torch.rand(4, device=device)
a = torch.cos(0.5 * math.pi * t)[:, None, None]
s = torch.sin(0.5 * math.pi * t)[:, None, None]
pred_noise = dp(a * clean + s * noise, obs, t)
F.mse_loss(pred_noise, noise).backward()
print("Diffusion rollout:", sample_diffusion(dp, obs[:1]).shape)
print("U-Net grad:", dp.middle.c1.weight.grad.norm().item())


## 3. π0-style — separate VLM/action expert → asymmetric attention → flow rollout


In [ ]:
def pi0_mask(prefix, suffix, device):
    mask = torch.zeros(prefix + suffix, prefix + suffix, dtype=torch.bool, device=device)
    mask[:prefix, :prefix] = True
    mask[prefix:, :] = True
    return mask

class AdaRMS(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.norm, self.scale, self.shift = nn.RMSNorm(d), nn.Linear(d, d), nn.Linear(d, d)

    def forward(self, x, c):
        return self.norm(x) * (1 + self.scale(c)[:, None]) + self.shift(c)[:, None]

class Pi0Block(nn.Module):
    def __init__(self, d=24, heads=3):
        super().__init__()
        self.heads, self.hd = heads, d // heads
        self.pnorm, self.enorm = nn.RMSNorm(d), AdaRMS(d)
        self.pqkv = nn.Linear(d, 3 * d, bias=False)
        self.eqkv = nn.Linear(d, 3 * d, bias=False)
        self.pout, self.eout = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.pff = nn.Sequential(nn.RMSNorm(d), nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.effnorm = AdaRMS(d)
        self.eff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def qkv(self, x, proj):
        b, n, d = x.shape
        qkv = proj(x).view(b, n, 3, self.heads, self.hd)
        return qkv.permute(2, 0, 3, 1, 4).unbind(0)

    def forward(self, prefix, suffix, c):
        pq, pk, pv = self.qkv(self.pnorm(prefix), self.pqkv)
        eq, ek, ev = self.qkv(self.enorm(suffix, c), self.eqkv)
        q, k, v = (torch.cat(items, dim=2) for items in [(pq, eq), (pk, ek), (pv, ev)])
        mask = pi0_mask(prefix.size(1), suffix.size(1), prefix.device)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        cut = prefix.size(1)
        py = y[:, :, :cut].transpose(1, 2).contiguous().flatten(2)
        ey = y[:, :, cut:].transpose(1, 2).contiguous().flatten(2)
        prefix = prefix + self.pout(py)
        suffix = suffix + self.eout(ey)
        prefix = prefix + self.pff(prefix)
        suffix = suffix + self.eff(self.effnorm(suffix, c))
        return prefix, suffix

class TinyPi0(nn.Module):
    def __init__(self, d=24, horizon=6, action_dim=3, depth=3):
        super().__init__()
        self.horizon, self.action_dim = horizon, action_dim
        self.vision, self.language = nn.Linear(10, d), nn.Embedding(32, d)
        self.state, self.action = nn.Linear(6, d), nn.Linear(action_dim, d)
        self.time = nn.Sequential(nn.Linear(1, d), nn.SiLU(), nn.Linear(d, d))
        self.blocks = nn.ModuleList([Pi0Block(d, 3) for _ in range(depth)])
        self.velocity = nn.Linear(d, action_dim)

    def forward(self, vision, language, state, actions, t):
        prefix = torch.cat([self.vision(vision), self.language(language)], dim=1)
        c = self.time(t[:, None])
        suffix = torch.cat(
            [self.state(state).unsqueeze(1), c.unsqueeze(1), self.action(actions)], dim=1
        )
        for block in self.blocks:
            prefix, suffix = block(prefix, suffix, c)
        return self.velocity(suffix[:, -self.horizon:])

@torch.no_grad()
def sample_pi0(model, vision, language, state, steps=6):
    x = torch.randn(state.size(0), model.horizon, model.action_dim, device=state.device)
    for step in range(steps, 0, -1):
        t = torch.full((state.size(0),), step / steps, device=state.device)
        x = x - model(vision, language, state, x, t) / steps
    return x

pi0 = TinyPi0().to(device)
vision = torch.randn(3, 3, 10, device=device)
language = torch.randint(0, 32, (3, 4), device=device)
state = torch.randn(3, 6, device=device)
actions = torch.randn(3, 6, 3, device=device)
noise = torch.randn_like(actions)
t = torch.rand(3, device=device)
xt = (1 - t[:, None, None]) * actions + t[:, None, None] * noise
F.mse_loss(pi0(vision, language, state, xt, t), noise - actions).backward()
print("π0 rollout:", sample_pi0(pi0, vision[:1], language[:1], state[:1]).shape)
print("expert grad:", pi0.blocks[0].eqkv.weight.grad.norm().item())


## 4. FAST — actual BPE encode/decode round trip

The check decodes the **BPE token stream itself**, then reconstructs quantized DCT coefficients
before inverse DCT. Reconstructing from the pre-BPE tensor would not test FAST tokenization.


In [ ]:
def dct_matrix(n, device):
    i = torch.arange(n, device=device, dtype=torch.float32)
    k = torch.arange(n, device=device, dtype=torch.float32)[:, None]
    dct = torch.cos(math.pi / n * (i + 0.5) * k)
    dct[0] *= math.sqrt(1 / n)
    dct[1:] *= math.sqrt(2 / n)
    return dct

def bounds(data):
    flat = data.reshape(-1, data.size(-1))
    return torch.quantile(flat, 0.01, dim=0), torch.quantile(flat, 0.99, dim=0)

def normalize(data, low, high):
    return (2 * (data - low) / (high - low).clamp_min(1e-6) - 1).clamp(-1, 1)

def denormalize(data, low, high):
    return 0.5 * (data + 1) * (high - low) + low

def merge_once(seq, pair, token):
    out, i = [], 0
    while i < len(seq):
        if i + 1 < len(seq) and (seq[i], seq[i + 1]) == pair:
            out.append(token)
            i += 2
        else:
            out.append(seq[i])
            i += 1
    return out

def pair_counts(seq):
    return Counter(zip(seq[:-1], seq[1:])) if len(seq) > 1 else Counter()

def train_bpe(corpus, merges=8):
    corpus = [list(seq) for seq in corpus]
    next_token = max(max(seq) for seq in corpus if seq) + 1
    rules = []
    for _ in range(merges):
        counts = Counter()
        for seq in corpus:
            counts.update(pair_counts(seq))
        if not counts:
            break
        pair, count = counts.most_common(1)[0]
        if count < 2:
            break
        rules.append((next_token, pair))
        corpus = [merge_once(seq, pair, next_token) for seq in corpus]
        next_token += 1
    return rules

def bpe_encode(seq, rules):
    seq = list(seq)
    for token, pair in rules:
        seq = merge_once(seq, pair, token)
    return seq

def bpe_decode(seq, rules):
    table = dict(rules)
    def expand(token):
        if token not in table:
            return [token]
        left, right = table[token]
        return expand(left) + expand(right)
    decoded = []
    for token in seq:
        decoded.extend(expand(token))
    return decoded

time = torch.linspace(0, 1, 8, device=device)
pattern_a = torch.stack(
    [torch.sin(2 * math.pi * time), torch.cos(2 * math.pi * time), 2 * time - 1], -1
)
pattern_b = torch.stack(
    [torch.sin(4 * math.pi * time), torch.cos(4 * math.pi * time), 1 - 2 * time], -1
)
data = torch.stack([pattern_a if i % 2 == 0 else pattern_b for i in range(64)])
low, high = bounds(data)
dct = dct_matrix(data.size(1), device)
coeff = torch.einsum("kt,btd->bkd", dct, normalize(data, low, high))
quantized = torch.round(coeff * 64).long()
offset = -int(quantized.min().item())
symbol_tensor = quantized + offset
corpus = [sample.flatten().tolist() for sample in symbol_tensor]
rules = train_bpe(corpus)
scalar_symbols = corpus[0]
bpe_tokens = bpe_encode(scalar_symbols, rules)
decoded_symbols = bpe_decode(bpe_tokens, rules)
assert decoded_symbols == scalar_symbols
assert rules and len(bpe_tokens) < len(scalar_symbols)
decoded_q = torch.tensor(decoded_symbols, device=device).view_as(quantized[:1]) - offset
assert torch.equal(decoded_q, quantized[:1])
decoded_coeff = decoded_q.float() / 64
decoded_norm = torch.einsum("kt,bkd->btd", dct, decoded_coeff)
reconstructed = denormalize(decoded_norm, low, high)
print("scalar symbols:", len(scalar_symbols), "BPE tokens:", len(bpe_tokens))
print("merge rules:", len(rules), "BPE decode exact:", decoded_symbols == scalar_symbols)
print("quantized decode exact:", torch.equal(decoded_q, quantized[:1]))
print("action roundtrip MSE:", F.mse_loss(reconstructed, data[:1]).item())


## References and provenance

- ACT: CVAE latent and Transformer action-query chunk decoder.
- Diffusion Policy: observation-conditioned temporal U-Net and iterative denoising.
- π0-style: separate VLM/action-expert parameters, asymmetric joint attention, flow rollout.
- FAST: quantile normalization, DCT, scalar discretization, BPE compression, and actual BPE decode.
